# 03 · Configuração 3: especialização com LoRA
O LoRA ensina **comportamento** (JSON, citar o id certo, recusar quando o trecho não está no
contexto), não decora a lei. Treinado só com `avaliacao/lora_treino.jsonl` (disjunto do teste).
Para cada pergunta que consta geramos também um exemplo **sem** o trecho certo, cujo alvo é a recusa.

In [ ]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/SEU_USUARIO/assistente-acessibilidade-calcadas.git"  # <- troque pelo seu repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

In [ ]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE, ARQ_TREINO
from src.lora import montar_exemplos, treinar
trechos = carregar_trechos(); por_id = {t["id"]: t for t in trechos}
treino, teste = carregar_jsonl(ARQ_TREINO), carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido")
exemplos = montar_exemplos(treino, buscador, por_id)
print(len(exemplos), "exemplos de treino")
print(exemplos[0]["mensagens"][1]["content"][:600], "\n\nALVO:", exemplos[0]["alvo"])

In [ ]:
treinar(exemplos, epocas=3, lr=2e-4, r=16)

## Avaliação com o adaptador

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
from src.gerador import carregar_modelo
from src.experimento import rodar
modelo = carregar_modelo(com_lora=True); print(modelo.nome)
rag_lora = rodar("rag_lora", teste, modelo, buscador, carregar_limiar())
for s in rag_lora[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

O adaptador fica em `modelos/lora-acessibilidade/` (~20 MB) e é versionado no repositório.